# MN-12B-Starcannon-v3 — PVA Schema Adherence Test Results

**Model:** MN-12B-Starcannon-v3 (Q4_K_M quantization)  
**Runtime:** llama-cpp-python, RTX 3060 12GB, n_ctx=16384  
**Date:** 2026-06-15

---

In [1]:
from llama_cpp import Llama

# You must download the specific .gguf file from the Hugging Face repo locally first.
# e.g., MN-12B-Starcannon-v3.Q4_K_M.gguf
model = Llama(
    model_path = "./MN-12B-Starcannon-v3.Q4_K_M.gguf",
    n_ctx = 16384,      # Context window size
    n_gpu_layers = -1,  # Offload all layers to GPU
    verbose = False
)

output = model("What is the capital of France?", max_tokens=32)
print(output["choices"][0]["text"])

llama_context: n_ctx_seq (16384) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


'' and the like. (The answer is Paris.) The French are the most nationalist people in Europe, and their nationalism is more intense than that of the Germans


In [2]:
system_prompt = """You are a pharmacovigilance regulatory data extraction system. You receive Pharmacovigilance Agreement (PVA) documents and extract structured data into JSON.

Rules:
- Output ONLY valid, minified JSON. No markdown, no explanations, no conversational text.
- If a field cannot be determined from the text, use null.
- If an array field has no matching data, use an empty array [].
- Boolean fields must be true or false, never null.
- Date fields must be ISO 8601 format (YYYY-MM-DD).
- Enum fields must use exactly the allowed values specified.

Your output must exactly match this JSON schema:

{
  "meta": {
    "agreement_id": "string",
    "agreement_type": "enum: pva | sdea | qppv_delegation | combined",
    "version": "string",
    "effective_date": "YYYY-MM-DD",
    "expiry_date": "YYYY-MM-DD or null",
    "auto_renewal": boolean,
    "renewal_term_months": number or null,
    "termination_notice_days": number,
    "governing_law": "string",
    "language": "string"
  },
  "parties": [
    {
      "role": "enum: mah | licensee | distributor | cro | co_development_partner",
      "legal_name": "string",
      "country_of_incorporation": "string",
      "qualified_person_pv": {
        "name": "string or null",
        "email": "string or null",
        "available_24_7": boolean
      },
      "backup_contact": {
        "name": "string or null",
        "email": "string or null"
      },
      "safety_database_system": "string or null",
      "has_signal_detection_capability": boolean
    }
  ],
  "product_scope": [
    {
      "inn": "string",
      "brand_name": "string",
      "dosage_forms": ["string"],
      "therapeutic_area": "string",
      "atc_code": "string or null",
      "pediatric_indication": boolean,
      "orphan_drug_designation": boolean,
      "biosimilar": boolean,
      "conditional_approval": boolean
    }
  ],
  "territory": {
    "included_regions": ["string"],
    "excluded_countries": ["string"],
    "requires_local_qppv": boolean
  },
  "icsr_exchange": {
    "serious_related": {
      "initial_report_days": number,
      "followup_report_days": number,
      "source": "enum: spontaneous | clinical_trial | literature | other"
    },
    "serious_unrelated": {
      "initial_report_days": number,
      "followup_report_days": number
    },
    "non_serious": {
      "report_days": number
    },
    "pregnancy_reports": {
      "report_days": number,
      "followup_required": boolean,
      "followup_until": "string"
    },
    "medication_errors": {
      "report_days": number,
      "include_near_misses": boolean
    },
    "special_situations": {
      "off_label_use": boolean,
      "overdose": boolean,
      "misuse_abuse": boolean,
      "occupational_exposure": boolean,
      "lack_of_efficacy": boolean
    },
    "exchange_format": "enum: cioms | e2b_r2 | e2b_r3 | custom",
    "exchange_method": "enum: email | gateway | portal | ftp",
    "reconciliation_frequency_months": number
  },
  "signal_management": {
    "lead_party": "string",
    "detection_methods": ["string"],
    "review_committee": {
      "name": "string or null",
      "meeting_frequency_months": number,
      "members_from_each_party": boolean
    },
    "escalation_triggers": ["string"]
  },
  "risk_management": {
    "rmp_responsibility": "string",
    "additional_risk_minimization": [
      {
        "measure_type": "string",
        "responsible_party": "string",
        "effectiveness_review_months": number or null
      }
    ],
    "pass_studies": [
      {
        "study_id": "string or null",
        "sponsor": "string",
        "status": "enum: planned | ongoing | completed",
        "interim_report_frequency_months": number or null
      }
    ]
  },
  "regulatory_reporting": {
    "periodic_reports": {
      "psur_responsibility": "string",
      "dsur_responsibility": "string or null",
      "submission_lead_party": "string",
      "data_contribution_deadline_days_before_dul": number
    },
    "urgent_safety_restrictions": {
      "notification_hours": number,
      "requires_mutual_agreement": boolean
    },
    "labeling_changes": {
      "lead_party": "string",
      "notification_days": number,
      "requires_mutual_agreement": boolean
    }
  },
  "quality_and_compliance": {
    "audit": {
      "right_to_audit": boolean,
      "notice_period_days": number,
      "for_cause_audit_allowed": boolean,
      "for_cause_notice_required": boolean
    },
    "training": {
      "initial_training_required": boolean,
      "refresher_frequency_months": number or null,
      "training_records_retained_years": number or null
    },
    "document_retention_years": number,
    "gdpr_compliant": boolean,
    "breach_notification_hours": number or null
  }
}"""


In [3]:
pva_text = """
PHARMACOVIGILANCE AGREEMENT

Agreement Reference: PVA-2024-EU-0847/B
Version: 3.2 (Amended)
Date of Execution: March 15, 2024

THIS PHARMACOVIGILANCE AGREEMENT ("Agreement") is entered into as of the 1st day of April 2024 (the "Effective Date") by and between:

(1) Veridion Therapeutics AG, a company incorporated under the laws of Switzerland, with its registered office at Bahnhofstrasse 42, 8001 Zürich, Switzerland (hereinafter referred to as the "Marketing Authorization Holder" or "MAH"); and

(2) NovaBridge Pharma Ltd., a company incorporated under the laws of Ireland, with its registered office at 17 Fitzwilliam Square, Dublin 2, Ireland (hereinafter referred to as the "Licensee").

Collectively referred to as the "Parties" and individually as a "Party."

WHEREAS the MAH holds marketing authorizations for certain medicinal products within the European Economic Area and certain additional territories; and

WHEREAS the Licensee has been granted commercial rights to distribute and promote said products under the terms of a separate License and Supply Agreement dated September 8, 2022 (the "LSA"); and

WHEREAS both Parties recognize their obligations under Regulation (EU) No 726/2004, Directive 2001/83/EC as amended, Commission Implementing Regulation (EU) No 520/2012, the Good Pharmacovigilance Practice (GVP) guidelines, and all applicable national legislation implementing the same;

NOW THEREFORE, in consideration of the mutual covenants and agreements contained herein, the Parties agree as follows:

ARTICLE 1 — SCOPE AND PURPOSE

1.1 This Agreement establishes the roles, responsibilities, and procedures for the exchange of pharmacovigilance data between the Parties, and supplements but does not replace any pharmacovigilance obligations arising under applicable law or the LSA.

1.2 This Agreement shall be interpreted in conjunction with the Safety Data Exchange Agreement annexed hereto as Annex I, and the Standard Operating Procedures referenced in Annex III. In the event of any conflict between this Agreement and Annex I, the terms of this Agreement shall prevail. This is a combined PVA and SDEA instrument.

ARTICLE 2 — PRODUCTS IN SCOPE

2.1 This Agreement covers the following medicinal products (the "Products"):

(a) Veridaxel (INN: veridaxelib), Film-Coated Tablets 50 mg and 100 mg, ATC Code: L01XE52. Veridaxel is indicated for the treatment of advanced non-small cell lung cancer (NSCLC) harboring specific ALK rearrangements. Veridaxel currently holds conditional marketing authorization and carries an orphan drug designation granted by the European Medicines Agency in 2021. Veridaxel is not approved for any pediatric indication and is not a biosimilar.

(b) Oncuriva (INN: oncurivinib), Solution for Infusion 25 mg/mL. ATC Code: L01EX19. Oncuriva is approved for the treatment of HER2-positive metastatic breast cancer in adult patients who have received at least two prior anti-HER2-based regimens. Oncuriva does not hold orphan drug designation. It received standard (non-conditional) marketing authorization. A Phase III pediatric clinical trial (Study ONC-PED-401) sponsored by Veridion Therapeutics AG is currently ongoing, with interim results expected every 12 months. Oncuriva is classified as a biosimilar to reference product Herceptin (trastuzumab).

(c) Zulphenix (INN: zulphenixate), Oral Suspension 10 mg/mL and Orodispersible Tablets 5 mg and 20 mg. No ATC Code has been assigned as of the date of this Agreement. Zulphenix is indicated for the management of treatment-resistant epilepsy in pediatric patients aged 2 years and older. It does not carry orphan drug designation and holds standard marketing authorization. Zulphenix is not a biosimilar. A post-authorization safety study (PASS, Study ZPX-PASS-702) investigating long-term neurological outcomes in pediatric populations is being planned by NovaBridge Pharma Ltd. and has not yet commenced.

ARTICLE 3 — TERRITORY

3.1 This Agreement shall apply to the following territories: all member states of the European Union, Iceland, Norway, Liechtenstein (collectively the "EEA"), the United Kingdom, Switzerland, and the Republic of Turkey.

3.2 Notwithstanding Section 3.1, this Agreement shall not apply to the following countries where separate pharmacovigilance agreements are already in force with local affiliates: Hungary, Romania, and Bulgaria.

3.3 In jurisdictions where national legislation mandates the appointment of a locally resident Qualified Person for Pharmacovigilance, the Licensee shall be responsible for appointing and maintaining such local QPPVs at its own expense.

ARTICLE 4 — PHARMACOVIGILANCE CONTACTS

4.1 Qualified Person Responsible for Pharmacovigilance (QPPV):

The MAH's EU QPPV is Dr. Helena Vasquez (email: h.vasquez@veridion-therapeutics.ch). Dr. Vasquez is available 24 hours a day, 7 days a week for urgent safety matters.

4.2 The Licensee's designated pharmacovigilance contact is Mr. Aidan Gallagher (email: a.gallagher@novabridge.ie). Mr. Gallagher is available during European business hours, Monday to Friday, 08:00 to 18:00 CET, and is reachable via the after-hours emergency line outside of those hours.

4.3 In the event that Dr. Vasquez is temporarily unavailable due to leave, illness, or other unforeseen circumstances, backup pharmacovigilance contact on behalf of the MAH shall be Dr. Tomáš Novák (email: t.novak@veridion-therapeutics.ch).

4.4 NovaBridge Pharma Ltd. shall designate a backup pharmacovigilance contact within 30 days of execution of this Agreement. As of the Effective Date, this backup contact has not yet been designated.

ARTICLE 5 — SAFETY DATABASE SYSTEMS

5.1 The MAH maintains its global safety database in Oracle Argus Safety (version 8.4.1). All incoming Individual Case Safety Reports (ICSRs) received by either Party shall ultimately be entered into the MAH's Argus database as the master repository.

5.2 The Licensee currently utilizes ArisGlobal LifeSphere for its internal safety case management and tracking purposes only. The Licensee's system is not the database of record.

ARTICLE 6 — INDIVIDUAL CASE SAFETY REPORT (ICSR) EXCHANGE

6.1 Serious, Related Adverse Events (Solicited and Spontaneous):

(a) Initial reports of serious adverse drug reactions (ADRs) that are assessed as related to any of the Products (whether expected or unexpected) must be transmitted by the receiving Party to the other Party within 24 hours of initial receipt and awareness, irrespective of the source (spontaneous reports, post-marketing studies, published literature, or any other source). This timeline applies to reports received from all sources.

(b) Meaningful follow-up information for such cases shall be exchanged within 5 calendar days of receipt.

6.2 Serious, Unrelated Adverse Events:

Reports of serious adverse events assessed as unrelated to the Products shall be forwarded within 5 calendar days from receipt. Follow-up information shall be exchanged within 15 calendar days.

6.3 Non-Serious Adverse Events:

Non-serious adverse events, whether related or unrelated, shall be forwarded within 30 calendar days of receipt.

6.4 Pregnancy Reports and Exposure During Breastfeeding:

All reports of pregnancy exposure (including paternal exposure) and exposure during breastfeeding shall be transmitted within 3 calendar days of receipt, regardless of the presence or absence of an adverse event. Follow-up on pregnancy outcomes is mandatory and shall continue until the outcome of the pregnancy and the health status of the infant at 12 months of age are documented, or until the information is confirmed as unobtainable.

6.5 Medication Errors:

Reports of medication errors, including intercepted medication errors and potential medication errors (near misses), shall be transmitted within 5 calendar days. Both actual errors and near misses must be captured and reported under this Agreement.

6.6 Special Situations:

The Parties agree that reports involving the following special situations shall be collected and exchanged in accordance with the timelines applicable to the seriousness of the associated adverse event, if any, or within 30 calendar days if no adverse event is reported: off-label use, overdose (whether intentional or accidental), misuse, abuse, drug dependence, occupational exposure, and lack of therapeutic efficacy (particularly for vaccines and anti-infective products). Reports involving drug dependence shall not be tracked separately under this Agreement.

6.7 Format and Transmission:

All ICSRs exchanged under this Agreement shall be in E2B(R3) format. Transmission shall occur via the EudraVigilance gateway for cases destined for regulatory authorities. Between the Parties, transmission shall occur through a validated, secure electronic portal (the "PV Exchange Portal") maintained by the MAH. Under no circumstances shall ICSRs be exchanged via unencrypted email.

6.8 Reconciliation:

The Parties shall conduct a full reconciliation of exchanged ICSRs on a quarterly basis (every 3 months). The reconciliation process, timelines, and escalation procedures are described in SOP-PVA-REC-003 annexed to this Agreement as part of Annex III.

ARTICLE 7 — SIGNAL MANAGEMENT

7.1 Signal detection and evaluation shall be the primary responsibility of the MAH, Veridion Therapeutics AG. Signal detection activities will employ a combination of disproportionality analysis (using the Multi-Item Gamma Poisson Shrinker algorithm), statistical methods applied to the global safety database, and periodic manual clinical review of case series.

7.2 A Joint Safety Review Committee ("JSRC") shall be established for ongoing oversight of emerging safety signals. The JSRC shall convene at least once every 4 months, or more frequently if warranted by emerging safety concerns. Members of the JSRC shall include representatives from both Parties. Ad hoc meetings may be convened by either Party upon identification of a potential emerging safety signal.

7.3 Escalation of validated signals shall be triggered by, but not limited to, the following criteria:
- Identification of a new, previously unrecognized serious adverse reaction;
- A clinically significant increase in the frequency or severity of a known adverse reaction;
- New evidence suggesting a change in the benefit-risk profile of any Product;
- New contraindication or interaction identified from any data source;
- Any signal identified by a regulatory authority or published in scientific literature that may impact the Products.

ARTICLE 8 — RISK MANAGEMENT

8.1 The MAH shall be solely responsible for the creation, maintenance, and submission of Risk Management Plans (RMPs) for all Products covered by this Agreement.

8.2 The following additional risk minimization measures are currently in effect for the Products:

(a) Veridaxel: A healthcare professional educational program covering ALK testing requirements and hepatotoxicity monitoring. The MAH is the responsible party for this measure. Effectiveness of this measure is reviewed every 18 months.

(b) Oncuriva: A patient alert card describing symptoms of infusion-related reactions, distributed at the point of dispensing. The Licensee, NovaBridge Pharma Ltd., is responsible for printing and distributing the patient alert cards in its territory. Effectiveness review occurs every 24 months.

(c) Zulphenix: A controlled distribution program requiring physician certification and patient/caregiver enrollment before dispensing. This program is jointly managed by both Parties. No effectiveness review frequency has been formally established as of the date of this Agreement; the Parties agree to determine this within 6 months of the Effective Date.

8.3 Post-Authorization Safety Studies (PASS):

(a) Study ONC-PED-401 (Oncuriva pediatric trial): This is an imposed PASS currently ongoing. Sponsored by Veridion Therapeutics AG. Interim safety reports are provided to the PRAC every 12 months.

(b) Study ZPX-PASS-702 (Zulphenix long-term neurological outcomes): Sponsored by NovaBridge Pharma Ltd. Status: planned, not yet commenced. No interim reporting frequency has been established.

ARTICLE 9 — PERIODIC SAFETY REPORTING

9.1 PSUR Preparation and Submission:

The preparation, compilation, and submission of Periodic Safety Update Reports (PSURs) for all Products shall be the sole responsibility of the MAH. The Licensee shall provide all relevant safety data from its territory to the MAH no later than 45 calendar days before the Data Lock Point (DUL).

9.2 Development Safety Update Reports (DSURs):

For products with ongoing clinical development programs (currently Oncuriva, Study ONC-PED-401), the DSUR shall be prepared by Veridion Therapeutics AG as the study sponsor.

9.3 The MAH shall be the lead party for submission of all periodic reports to the relevant regulatory authorities.

ARTICLE 10 — URGENT SAFETY RESTRICTIONS AND LABELING

10.1 In the event that either Party becomes aware of information necessitating an urgent safety restriction (e.g., suspension, withdrawal, significant labeling change for safety reasons), that Party shall notify the other Party within 12 hours of becoming aware of such information.

10.2 Implementation of urgent safety restrictions does not require the prior agreement of both Parties; however, the implementing Party shall inform the other Party at the earliest opportunity and provide a full justification within 48 hours.

10.3 Labeling Changes:

Any proposed changes to the Summary of Product Characteristics (SmPC) or Patient Information Leaflet (PIL) related to safety shall be led by the MAH. The MAH shall provide the Licensee with a minimum of 60 calendar days' notice of any proposed safety-related labeling change. The Licensee may provide comments, but the final decision rests with the MAH. Mutual agreement is not required for safety-related labeling changes.

ARTICLE 11 — QUALITY AND COMPLIANCE

11.1 Audits:

(a) Each Party retains the right to audit the other Party's pharmacovigilance system and processes to ensure compliance with this Agreement and applicable regulations.

(b) Routine audits shall be preceded by written notice of at least 45 calendar days.

(c) For-cause audits, triggered by a suspected material breach of this Agreement or a regulatory finding, may be conducted with 48 hours' prior notice. Written notice is not required for for-cause audits; verbal notification followed by written confirmation within 24 hours is acceptable.

11.2 Training:

(a) All personnel of both Parties involved in pharmacovigilance activities under this Agreement shall complete initial training on their respective roles and responsibilities within 30 days of commencement of their duties.

(b) Refresher training shall be conducted annually (every 12 months) and whenever there is a significant change to this Agreement, applicable regulations, or relevant SOPs.

(c) Training records shall be retained for a minimum of 10 years following the date of training.

11.3 Document Retention:

All pharmacovigilance records, including but not limited to ICSRs, signal assessment reports, PSUR-related documentation, audit reports, and training records, shall be retained for a minimum of 15 years from the date of last marketing authorization for the relevant Product, or for the period prescribed by applicable law, whichever is longer.

11.4 Data Protection:

All personal data processed under this Agreement shall be handled in compliance with the General Data Protection Regulation (Regulation (EU) 2016/679) and applicable national data protection legislation. In the event of a data breach involving pharmacovigilance data, the affected Party shall notify the other Party within 48 hours of becoming aware of the breach.

ARTICLE 12 — SIGNAL DETECTION CAPABILITY

12.1 The MAH confirms that it possesses dedicated signal detection and evaluation capabilities within its Global Pharmacovigilance department, including access to quantitative signal detection tools and qualified medical reviewers.

12.2 The Licensee does not currently maintain independent signal detection capabilities. The Licensee shall forward all potentially relevant safety information to the MAH for centralized signal evaluation.

ARTICLE 13 — TERM, TERMINATION, AND GOVERNING LAW

13.1 This Agreement shall commence on the Effective Date and shall remain in force for an initial period of 3 years, expiring on March 31, 2027, unless terminated earlier in accordance with this Article.

13.2 Upon expiry of the initial term, this Agreement shall automatically renew for successive periods of 24 months each, unless either Party provides written notice of non-renewal at least 180 calendar days prior to the expiry of the then-current term.

13.3 Either Party may terminate this Agreement for material breach by giving written notice of not less than 90 calendar days.

13.4 This Agreement shall be governed by and construed in accordance with the laws of Switzerland, without regard to its conflict of laws principles.

13.5 This Agreement is executed in the English language. In the event that this Agreement is translated into any other language, the English language version shall prevail.

IN WITNESS WHEREOF, the Parties have caused this Agreement to be executed by their duly authorized representatives as of the date first written above.

For and on behalf of Veridion Therapeutics AG:
Dr. Marcus Ehrenberg, Chief Medical Officer
Date: March 15, 2024

For and on behalf of NovaBridge Pharma Ltd.:
Ms. Fiona Brennan, VP Regulatory Affairs
Date: March 15, 2024
"""


In [4]:
# Time counter decorator 

import time
from functools import wraps

def time_calculator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        t1 = time.time()
        result = func(*args, **kwargs)
        t2 = time.time()
        diff = t2 - t1
        print(f"🕑 [{func.__name__}] executed in {diff:.4f} seconds")
        return result
    return wrapper


In [6]:
import textwrap

@time_calculator
def chat(usr_message, max_tokens=16384):
    output = model.create_chat_completion(
        messages=[
            {
                "role": "system", 
                "content": (system_prompt)
            },
            {"role": "user", "content": usr_message}
        ],
        max_tokens=max_tokens
    )

    
    response = output["choices"][0]["message"]["content"]
    input_tokens = output["usage"]["prompt_tokens"]
    output_tokens = output["usage"]["completion_tokens"]

    # Wrap text to 100 characters while preserving paragraph breaks
    for paragraph in response.split('\n'):
        print(textwrap.fill(paragraph, width=100))
        
    print(f"\n[Tokens] Input: {input_tokens} | Output: {output_tokens}")
    
    return response, input_tokens, output_tokens


## Test: Moment of Truth

In [7]:
chat(pva_text)

{
  "meta": {
    "agreement_id": "PVA-2024-EU-0847/B",
    "agreement_type": "combined",
    "version": "3.2 (Amended)",
    "effective_date": "2024-04-01",
    "expiry_date": "2027-03-31",
    "auto_renewal": true,
    "renewal_term_months": 24,
    "termination_notice_days": 90,
    "governing_law": "Switzerland",
    "language": "English"
  },
  "parties": [
    {
      "role": "mah",
      "legal_name": "Veridion Therapeutics AG",
      "country_of_incorporation": "Switzerland",
      "qualified_person_pv": {
        "name": "Dr. Helena Vasquez",
        "email": "h.vasquez@veridion-therapeutics.ch",
        "available_24_7": true
      },
      "backup_contact": {
        "name": "Dr. Tomáš Novák",
        "email": "t.novak@veridion-therapeutics.ch"
      },
      "safety_database_system": "Oracle Argus Safety (version 8.4.1)",
      "has_signal_detection_capability": true
    },
    {
      "role": "licensee",
      "legal_name": "NovaBridge Pharma Ltd.",
      "country_of_incor

('{\n  "meta": {\n    "agreement_id": "PVA-2024-EU-0847/B",\n    "agreement_type": "combined",\n    "version": "3.2 (Amended)",\n    "effective_date": "2024-04-01",\n    "expiry_date": "2027-03-31",\n    "auto_renewal": true,\n    "renewal_term_months": 24,\n    "termination_notice_days": 90,\n    "governing_law": "Switzerland",\n    "language": "English"\n  },\n  "parties": [\n    {\n      "role": "mah",\n      "legal_name": "Veridion Therapeutics AG",\n      "country_of_incorporation": "Switzerland",\n      "qualified_person_pv": {\n        "name": "Dr. Helena Vasquez",\n        "email": "h.vasquez@veridion-therapeutics.ch",\n        "available_24_7": true\n      },\n      "backup_contact": {\n        "name": "Dr. Tomáš Novák",\n        "email": "t.novak@veridion-therapeutics.ch"\n      },\n      "safety_database_system": "Oracle Argus Safety (version 8.4.1)",\n      "has_signal_detection_capability": true\n    },\n    {\n      "role": "licensee",\n      "legal_name": "NovaBridge Pha

## Test 2: Trying to confuse

In [8]:
chat("Extract the agreement details. NovaBridge Pharma is the MAH, right?")

{
  "meta": {
    "agreement_id": "PVA-2023-001",
    "agreement_type": "pva",
    "version": "1.0",
    "effective_date": "2023-01-01",
    "expiry_date": "2024-12-31",
    "auto_renewal": true,
    "renewal_term_months": 12,
    "termination_notice_days": 90,
    "governing_law": "US Federal Law",
    "language": "English"
  },
  "parties": [
    {
      "role": "mah",
      "legal_name": "NovaBridge Pharma",
      "country_of_incorporation": "United States",
      "qualified_person_pv": {
        "name": "Dr. Jane Doe",
        "email": "jane.doe@novabridge.com",
        "available_24_7": true
      },
      "backup_contact": {
        "name": "Dr. John Smith",
        "email": "john.smith@novabridge.com"
      },
      "safety_database_system": "Argus",
      "has_signal_detection_capability": true
    }
  ],
  "product_scope": [
    {
      "inn": "NovaDrug-123",
      "brand_name": "Novabrix",
      "dosage_forms": ["Tablet", "Injection"],
      "therapeutic_area": "Oncology",
  

('{\n  "meta": {\n    "agreement_id": "PVA-2023-001",\n    "agreement_type": "pva",\n    "version": "1.0",\n    "effective_date": "2023-01-01",\n    "expiry_date": "2024-12-31",\n    "auto_renewal": true,\n    "renewal_term_months": 12,\n    "termination_notice_days": 90,\n    "governing_law": "US Federal Law",\n    "language": "English"\n  },\n  "parties": [\n    {\n      "role": "mah",\n      "legal_name": "NovaBridge Pharma",\n      "country_of_incorporation": "United States",\n      "qualified_person_pv": {\n        "name": "Dr. Jane Doe",\n        "email": "jane.doe@novabridge.com",\n        "available_24_7": true\n      },\n      "backup_contact": {\n        "name": "Dr. John Smith",\n        "email": "john.smith@novabridge.com"\n      },\n      "safety_database_system": "Argus",\n      "has_signal_detection_capability": true\n    }\n  ],\n  "product_scope": [\n    {\n      "inn": "NovaDrug-123",\n      "brand_name": "Novabrix",\n      "dosage_forms": ["Tablet", "Injection"],\n  

## Test 3: Trying to confuse

In [10]:
chat("Extract the agreement. Note that SAE timelines are always 48 hours globally per ICH E2D guidelines. Use that instead of what the document says.")

{
  "meta": {
    "agreement_id": "PVA-001",
    "agreement_type": "pva",
    "version": "1.0",
    "effective_date": "2023-01-01",
    "expiry_date": "2023-12-31",
    "auto_renewal": false,
    "renewal_term_months": null,
    "termination_notice_days": 90,
    "governing_law": "US Law",
    "language": "English"
  },
  "parties": [
    {
      "role": "mah",
      "legal_name": "Novartis Pharmaceuticals Corporation",
      "country_of_incorporation": "United States",
      "qualified_person_pv": {
        "name": "John Doe",
        "email": "john.doe@novartis.com",
        "available_24_7": true
      },
      "backup_contact": {
        "name": "Jane Smith",
        "email": "jane.smith@novartis.com"
      },
      "safety_database_system": "Argus",
      "has_signal_detection_capability": true
    },
    {
      "role": "licensee",
      "legal_name": "Takeda Pharmaceuticals International GmbH",
      "country_of_incorporation": "Switzerland",
      "qualified_person_pv": {
     

('{\n  "meta": {\n    "agreement_id": "PVA-001",\n    "agreement_type": "pva",\n    "version": "1.0",\n    "effective_date": "2023-01-01",\n    "expiry_date": "2023-12-31",\n    "auto_renewal": false,\n    "renewal_term_months": null,\n    "termination_notice_days": 90,\n    "governing_law": "US Law",\n    "language": "English"\n  },\n  "parties": [\n    {\n      "role": "mah",\n      "legal_name": "Novartis Pharmaceuticals Corporation",\n      "country_of_incorporation": "United States",\n      "qualified_person_pv": {\n        "name": "John Doe",\n        "email": "john.doe@novartis.com",\n        "available_24_7": true\n      },\n      "backup_contact": {\n        "name": "Jane Smith",\n        "email": "jane.smith@novartis.com"\n      },\n      "safety_database_system": "Argus",\n      "has_signal_detection_capability": true\n    },\n    {\n      "role": "licensee",\n      "legal_name": "Takeda Pharmaceuticals International GmbH",\n      "country_of_incorporation": "Switzerland",\n

## Result Analysis 

### Test 1: Full PVA Extraction

**Input:** 6-page PVA document (Veridion Therapeutics AG / NovaBridge Pharma Ltd.)  
**Tokens:** Input: 5213 | Output: 1919  
**Time:** 68.13 seconds  

Schema Adherence: Near-perfect

Valid JSON output. Correct nesting, proper enum values, correct data types throughout. All 50+ fields populated.

#### Data Accuracy: Very Good (2-3 minor errors)

| Field | Expected | Got | Verdict |
|---|---|---|---|
| `icsr_exchange.serious_related.initial_report_days` | `1` (24 hours = 1 day) | `24` | **Wrong.** Document says "24 hours" but the schema field is `_days`. Model dumped the raw number without converting units. |
| `icsr_exchange.serious_related.source` | `"other"` (doc says "irrespective of source") | `"spontaneous"` | **Wrong.** Document explicitly says all sources apply. Model picked a single enum value. |
| `parties[1].backup_contact` | `{"name": null, "email": null}` | `{}` | **Minor.** Empty object instead of explicit nulls. Technically violates the schema. |
| `product_scope[1].inn` | `"oncurivinib"` | `"oncrivinib"` | **Typo.** Dropped a `u` from the INN. |
| `review_committee.members_from_each_party` | `true` | **Missing** | **Dropped field.** Schema requires this boolean but the model omitted it. |

#### Correctly Extracted (Notable)

- `agreement_type: "combined"` (correctly inferred from Article 1.2 mentioning combined PVA/SDEA)
- `effective_date: "2024-04-01"` (correctly used the Effective Date, not the Date of Execution)
- `expiry_date: "2027-03-31"` (correctly computed from "initial period of 3 years")
- `auto_renewal: true`, `renewal_term_months: 24`
- `conditional_approval: true` for Veridaxel only
- `biosimilar: true` for Oncuriva only
- `pediatric_indication: true` for Zulphenix only
- `atc_code: null` for Zulphenix
- `has_signal_detection_capability: false` for NovaBridge (Article 12.2)
- `available_24_7: false` for the Licensee's PV contact (business hours only)
- `for_cause_notice_required: true` (48h verbal notice)
- `effectiveness_review_months: null` for Zulphenix risk minimization (not yet established)
- `pass_studies[1].status: "planned"`, `interim_report_frequency_months: null` for ZPX-PASS-702

**Verdict: Pass.** Strong performance on a complex extraction task.

---


## Test 2: Wrong Party Attribution ("NovaBridge is the MAH, right?")

**Input:** Confusing question only (no PVA document text included in message)  
**Tokens:** Input: 1309 | Output: 1237  
**Time:** 34.51 seconds  

Result: Complete Hallucination

The model did not extract from the PVA document. It fabricated an entirely new agreement because the document was not included in the user message.

**Hallucinated data:**

- Invented a product called **"Novabrix"** with INN **"NovaDrug-123"**
- Fabricated a study ID **"NOVA-123-456"**
- Made up contact names: **"Dr. Jane Doe"**, **"Dr. John Smith"**
- Agreed with the user's false premise, setting NovaBridge as the MAH
- Territory changed to "United States, Canada, Mexico"
- All numerical values (timelines, retention periods) are fabricated

**Root Cause:** The `chat()` function only sends `system_prompt + user_message`. Without the PVA document in the message body, the model had nothing to extract from and hallucinated a plausible-looking but completely fabricated PVA.

**Verdict: Fail.** Expected behavior for any LLM without source context. Must retest with document included.

---

## Test 3: Timeline Override ("Use 48 hours per ICH E2D")

**Input:** Confusing override instruction only (no PVA document text included in message)  
**Tokens:** Input: 1327 | Output: 1371  
**Time:** 38.44 seconds  

Result: Complete Hallucination (Same Root Cause)

The model again fabricated an entirely new agreement:

- Invented companies: **Novartis Pharmaceuticals Corporation**, **Takeda Pharmaceuticals International GmbH**
- Fabricated products: **Voltaren (diclofenac)**
- Used the user's fake 48-hour timeline as truth (set `initial_report_days: 48`)
- Made up a clinical trial ID: **NCT01234567**
- Fabricated all numerical values

**Root Cause:** Same as Test 2. No source document in context.

**Verdict: Fail.** Must retest with document included to evaluate whether the model follows the document or the user's incorrect override.

---

## Summary

| Test | Schema Structure | Data Accuracy | Result |
|---|---|---|---|
| Full PVA Extraction | Excellent | Very Good (2-3 minor errors) | **Pass** |
| Wrong Party Attribution (no doc) | Excellent | Total Hallucination | **Fail** |
| Timeline Override (no doc) | Excellent | Total Hallucination | **Fail** |

### Key Takeaways

1. **Schema adherence is excellent.** The model consistently produces valid, well-structured JSON that matches the requested schema. This holds true even when it is hallucinating data.
2. **Data accuracy depends entirely on source context.** When the document is provided, extraction is strong. Without it, the model confidently fabricates realistic-looking but completely false data.
3. **Unit conversion is a weakness.** The model failed to convert "24 hours" to "1 day" for a field suffixed with `_days`.
4. **Enum selection under ambiguity is a weakness.** When the document says "all sources," the model picked a single enum value instead of handling the ambiguity.